<a href="https://colab.research.google.com/github/tiagobventura/ModeloCredito/blob/main/Sistemas_de_recomenda%C3%A7%C3%A3o_Python_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recomendação Simples

Carregando o conjunto de dados de metadados

In [6]:
import pandas as pd

metadata = pd.read_csv('/content/drive/MyDrive/Dataset/dataset recomendação filmes/movies_metadata.csv', low_memory=False)

metadata.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


\begin{equation} \text Weighted Rating (\bf WR) = \left({{\bf v} \over {\bf v} + {\bf m}} \cdot R\right) + \left({{\bf m} \over {\bf v} + {\bf m}} \cdot C\right) \end{equation}

Parâmetros da equação.
* v é o número de votos para o filme;
* m é o número mínimo de votos necessários para que você seja listado no gráfico;
* R é a classificação média do filme;
* C é o voto médio em todo o relatório.


In [3]:
#Classificação média dos filmes
C = metadata['vote_average'].mean()
print(C)

5.618207215134185


Na célula de cima, classificação média de um filme na base de dados é 5,6 em um escala de 10.

In [4]:
#Calculo do numero de votos é o m, recebidos por um filme no 90° percentil.
m = metadata['vote_count'].quantile(0.90)
print(m)

160.0


In [5]:
#Copiando a base de dados para realização de calculos, não afetando a base original.

q_movies = metadata.copy().loc[metadata['vote_count'] >= m]
q_movies.shape

(4555, 24)

In [6]:
metadata.shape

(45466, 24)

Abaixo, será calculado a classificação ponderada de cada filme qualificado.
O que será realizado nessa etapa:


*   Definir uma função weight_rating.
*  Já foi calculado o m e C, que serão passados como argumento para a função.
*  Será selecioando a coluna vote_count(v) e vote_average(R) do quadro do dataset q_movies.
* Por fim, será calculado a média ponderada e retornará o resultado.






In [7]:
#Função criada para calcular o valor.
def weighted_rating(x, m=m, C=C):
  v = x['vote_count']
  R = x['vote_average']

  #Calculo base na formula IMDB
  return (v/(v+m)*R) + (m/(m+v)*C)


In [12]:
#Criada a nova coluna 'score' que receberá o valor da função de classficação moderada.

q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

In [14]:
q_movies = q_movies.sort_values('score', ascending=False)

q_movies[['title', 'vote_count', 'vote_average', 'score']].head(20)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


# Recomendação baseado em conteúdo

In [7]:
metadata['overview'].head()

,overview
0,"Led by Woody, Andy's toys live happily in his ..."
1,When siblings Judy and Peter discover an encha...
2,A family wedding reignites the ancient feud be...
3,"Cheated on, mistreated and stepped on, the wom..."
4,Just when George Banks has recovered from his ...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
#Definindo um TF-IDF Vectorizer Object. Remove

#define a vetorização do objeto. Remove palavras como 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

metadata['overview'] = metadata['overview'].fillna('')

tfidf_matrix = tfidf.fit_transform(metadata['overview'])

tfidf_matrix.shape



(45466, 75827)

In [9]:
tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

In [ ]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)